# Améliorer la consigne : lettres doubles, lignes vides, zoom

**Notebook autonome**, construit sur les mêmes briques que le notebook 10 (Fiqual, divergences du SeRN,
vérité arbitrée, listes de l'Insee). Il ne change qu'une chose à la fois et mesure l'effet sur
**les mêmes lignes**.

## Les variantes testées

| Variante | Ce qui change | Ce qu'elle vise |
|---|---|---|
| **V0 actuelle** | la consigne du notebook 10 (lectures déjà faites, réutilisées) | référence |
| **V1 épeler** | nouvelle consigne : lettre par lettre, attention aux lettres doubles et aux cursives proches, réponse VIDE pour une ligne vide, épellation avant la réponse | le modèle qui devine le mot au lieu de compter les lettres |
| **V2 bloc + zoom** | même consigne que V1, mais l'image contient le bloc **et** un zoom ×2 des lignes Nom et Prénom | les lettres doubles perdues quand l'image est réduite |
| **V3 alerte lettres doubles** | sur la meilleure lecture : si la version avec (ou sans) une lettre doublée existe dans les listes et est au moins 3 fois plus portée, la ligne part en alerte| les erreurs qui donnent un autre nom existant |



In [1]:
# 1. Reglages et outils (repris du notebook 10)
import base64
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import zipfile
from pathlib import Path

import mlflow
import pandas as pd
import requests
from PIL import Image
from scipy import stats

CAMPAGNE = None                  # None = RG25 si elle a un fichier de divergences, sinon RG26
CAMPAGNES_POSSIBLES = ["RG25", "RG26"]
TEST_RAPIDE = False              # True : 10 bulletins (verification)
MODELE = "gemma4-26b-moe"
VARIABLE_NOM, VARIABLE_PRENOM = None, None     # None = detection automatique (colonne VARIABLE)
LONGUEUR_MIN_CORRECTION = 4
MAX_JETONS = 400                 # la consigne « epeler » demande une reponse plus longue
RATIO_DOUBLE = 3                 # alerte lettre double : variante au moins 3 fois plus portee
ZONES_LIGNES = {"NOM": (0.155, 0.083, 0.245, 0.011), "PRENOM": (0.155, 0.094, 0.245, 0.011)}

BUCKET = "projet-mesure-qualite-rp"
POS_UT, POS_CAB = 2, 4
COLS = {"NOM": 11, "PRENOM": 12}
BORDS_BLOC = (0.075, 0.072, 0.450, 0.105)
g, h, d_, b_ = BORDS_BLOC
ZONE_BLOC = (g, h, d_ - g, b_ - h)

RESULTATS = Path.home() / "work" / "banc-htr" / "resultats"
LISTES = Path.home() / "work" / "banc-htr" / "listes_insee"
BLOCS = Path("/tmp/arbitrage/blocs")
URL_NOMS = "https://www.insee.fr/fr/statistiques/fichier/3536630/noms2008nat_txt.zip"
URL_PRENOMS = "https://www.insee.fr/fr/statistiques/fichier/8595130/prenoms-2025-nat_csv.zip"
for d in [RESULTATS, LISTES, BLOCS]:
    d.mkdir(parents=True, exist_ok=True)
RACINE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
SUFFIXE = "_rapide" if TEST_RAPIDE else ""


def normaliser(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    t = unicodedata.normalize("NFKD", str(t).strip())
    t = "".join(c for c in t if not unicodedata.combining(c)).upper()
    t = re.sub(r"[^A-Z0-9 ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def distance(a, b):
    if a == b:
        return 0
    prec = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cour = [i]
        for j, cb in enumerate(b, 1):
            cour.append(min(prec[j] + 1, cour[j - 1] + 1, prec[j - 1] + (ca != cb)))
        prec = cour
    return prec[-1]


def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - m), min(1.0, c + m)


def ic(k, n):
    lo, hi = wilson(k, n)
    return f"[{lo * 100:.1f} ; {hi * 100:.1f}]"


def mcnemar(a_ok, b_ok):
    """(a seul juste, b seul juste, p exact)."""
    n10, n01 = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    return n10, n01, (stats.binomtest(n10, n10 + n01).pvalue if n10 + n01 else 1.0)


def manquant(v):
    return v is None or (isinstance(v, float) and math.isnan(v))


import s3fs
fs = s3fs.S3FileSystem()
print(f"mode {'TEST RAPIDE' if TEST_RAPIDE else 'COMPLET'}")

/home/onyxia/work/mesure-qualite-rp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mode COMPLET


In [2]:
# 2. Le fichier des divergences arbitrees par le SeRN
def fichiers_divergences(c):
    try:
        return sorted(f for f in fs.find(f"{BUCKET}/data/raw/{c}/Divergences") if f.lower().endswith((".xlsx", ".xls")))
    except FileNotFoundError:
        return []

disponibles = {c: fichiers_divergences(c) for c in CAMPAGNES_POSSIBLES}
CAMPAGNE = CAMPAGNE or next((c for c in CAMPAGNES_POSSIBLES if disponibles[c]), None)
if not CAMPAGNE or not disponibles.get(CAMPAGNE):
    raise RuntimeError("Aucun fichier de divergences trouve sur S3.")
morceaux = []
for f in disponibles[CAMPAGNE]:
    with fs.open(f, "rb") as fh:
        morceaux.append(pd.read_excel(io.BytesIO(fh.read()), dtype=str))
DIV = pd.concat(morceaux, ignore_index=True).fillna("")
DIV.columns = [re.sub(r"[\s\-]+", "_", c.strip().upper()) for c in DIV.columns]
manque = [c for c in ["VARIABLE", "NOM_IMAGE", "VALEUR_PAD", "VALEUR_PMQ", "VALEUR_ATTENDUE"] if c not in DIV.columns]
if manque:
    raise RuntimeError(f"Colonnes introuvables dans le fichier des divergences : {manque}")

vals = DIV.VARIABLE.str.upper().str.strip().unique()
VARIABLE_NOM = VARIABLE_NOM or next((v for v in vals if re.fullmatch(r"(NOM|NOMI|NOM_INDIV|NOMNAIS)", v)), None)
VARIABLE_PRENOM = VARIABLE_PRENOM or next((v for v in vals if re.fullmatch(r"(PRENOMS?|PRENOMI|PRENOM_INDIV)", v)), None)
if not VARIABLE_NOM or not VARIABLE_PRENOM:
    print(DIV.VARIABLE.value_counts().head(15).to_string())
    raise RuntimeError("Variables NOM / PRENOM non reconnues : les renseigner en cellule 1.")


def cle_divergence(nom_image):
    """Les deux morceaux juste avant 'BI' : '/99/976/602/0407_9761_5300000184_BI.jpg' -> '9761_5300000184'."""
    m = Path(str(nom_image)).stem.split("_")
    if m and m[-1].upper() == "BI":
        m = m[:-1]
    return "_".join(m[-2:])


DIV["cle"] = DIV.NOM_IMAGE.map(cle_divergence)
DIV["champ"] = DIV.VARIABLE.str.upper().str.strip().map({VARIABLE_NOM: "NOM", VARIABLE_PRENOM: "PRENOM"})
DIV_TOUT = DIV.copy()
DIV = DIV[DIV.champ.notna()].copy()
DIV["arbitree"] = DIV.VALEUR_ATTENDUE.str.strip() != ""
print(f"campagne {CAMPAGNE} | {len(DIV_TOUT)} divergences, dont {len(DIV)} sur NOM/PRENOM "
      f"({int(DIV.arbitree.sum())} arbitrees) | NOM = {VARIABLE_NOM!r}, PRENOM = {VARIABLE_PRENOM!r}")

campagne RG25 | 149 divergences, dont 22 sur NOM/PRENOM (5 arbitrees) | NOM = 'NOM', PRENOM = 'PRENOM'


In [3]:
# 3. Les bulletins, le Fiqual et les images des blocs
fichiers = [f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/BIE") if f.endswith("_BI.jpg")]
bi = pd.DataFrame({"chemin": fichiers})
m = bi.chemin.map(lambda c: Path(c).stem).str.split("_")
bi["cle"] = m.str[0] + "_" + m.str[1]
fiq = [f for f in fs.ls(f"{BUCKET}/data/raw/{CAMPAGNE}") if "FIQUAL" in Path(f).name.upper() and f.lower().endswith(".txt")]
with fs.open(fiq[0], "rb") as f:
    brut = pd.read_csv(f, sep="\x00", header=None, names=["ligne"], dtype=str,
                       encoding="latin-1", keep_default_na=False, engine="python")["ligne"]
ch = brut.str.split("|")
fq3 = pd.DataFrame(ch[ch.str[0] == "3"].tolist(), dtype=str)
fq3["cle"] = fq3[POS_UT].str.strip() + "_" + fq3[POS_CAB].str.strip()
ref = fq3[["cle", COLS["NOM"], COLS["PRENOM"]]].copy()
ref.columns = ["cle", "NOM", "PRENOM"]
N_FIQUAL = ref.cle.nunique()
APP = bi.merge(ref, on="cle", how="inner").drop_duplicates("cle").sort_values("cle").reset_index(drop=True)
APP = APP.head(10) if TEST_RAPIDE else APP
print(f"Fiqual : {Path(fiq[0]).name} | {len(APP)} bulletins apparies | "
      f"divergences NOM/PRENOM sur nos bulletins : {int(DIV.cle.isin(set(APP.cle)).sum())} sur {len(DIV)}")

for cz in [RACINE / "configs" / f"positions_BI_{CAMPAGNE}.json", RACINE / "notebooks" / "configs" / f"positions_BI_{CAMPAGNE}.json"]:
    if cz.exists() and "bloc" in json.loads(cz.read_text(encoding="utf-8")):
        g, h, d_, b_ = json.loads(cz.read_text(encoding="utf-8"))["bloc"]
        ZONE_BLOC = (g, h, d_ - g, b_ - h)
        break

Fiqual : RP_PMQ_FIQUAL2599.txt | 169 bulletins apparies | divergences NOM/PRENOM sur nos bulletins : 22 sur 22


In [4]:
# 4. Le lecteur gemma4, et la consigne ACTUELLE (lectures du notebook 09 reutilisees)
import numpy as np
import logging
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

N_PARALLELE = 4          # appels simultanes a gemma4 ; baisser a 2 si echecs ou lenteur
DELAI_S, TENTATIVES = 90, 2   # un appel bloque est abandonne en ~3 min (au lieu de 9)

BASE_URL = "https://llm.lab.sspcloud.fr/api"
PROMPT_BLOC = """Cette image montre un extrait de bulletin individuel du recensement francais.
Elle contient des lignes manuscrites precedees de leurs libelles imprimes :
"Nom :" et "Prenom :".

Transcris ce qui est ecrit a la main sur chacune, en MAJUSCULES, sans accent.

Regles :
- Ne corrige rien, n'ajoute rien, ne devine pas.
- Ne recopie pas les libelles imprimes, seulement l'ecriture manuscrite.
- Si une ligne n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {"NOM": "...", "PRENOM": "..."}"""
PROMPT_RELECTURE = """Cette image montre un extrait de bulletin individuel du recensement francais,
avec un nom et un prenom ecrits a la main apres les libelles imprimes "Nom :" et "Prenom :".

Une premiere lecture a donne : {lecture}
Ces valeurs ne figurent pas dans les listes de l'Insee des noms et prenoms portes en France.
Voici des valeurs existantes, proches de la premiere lecture :
{candidats}

Relis attentivement l'ecriture manuscrite.
- Si elle correspond vraiment a l'une de ces valeurs, reponds cette valeur.
- Sinon, recopie exactement ce qui est ecrit, meme si ce n'est pas dans la liste.
- Si ce n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {{"NOM": "...", "PRENOM": "..."}}"""
HEADERS = None

# --- Journal : progression, temps et erreurs (aucune lecture, donc aucun nom) ---
JOURNAL = RESULTATS / f"journal_arbitrage_{CAMPAGNE}{SUFFIXE}.log"
log = logging.getLogger("arbitrage")
log.setLevel(logging.INFO)
if not log.handlers:
    _h = logging.FileHandler(JOURNAL, encoding="utf-8")
    _h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    log.addHandler(_h)


def service_pret():
    global HEADERS
    if HEADERS:
        return
    if "OPENAI_API_KEY" not in os.environ and shutil.which("vault"):
        r = subprocess.run(["vault", "kv", "get", "-field=OPENAI_API_KEY", "onyxia-kv/farisazibert/LLM"],
                           capture_output=True, text=True, timeout=60)
        if r.returncode == 0 and r.stdout.strip():
            os.environ["OPENAI_API_KEY"] = r.stdout.strip()
    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("Cle du service absente : la renseigner (Vault).")
    HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}


def bloc(cle):
    fb = BLOCS / f"{CAMPAGNE}_{cle}.png"
    if not fb.exists():
        chemin = APP.loc[APP.cle == cle, "chemin"].iloc[0]
        with fs.open(chemin, "rb") as src:
            page = Image.open(io.BytesIO(src.read())).convert("RGB")
        L, H = page.size
        x, y, w, hh = ZONE_BLOC
        page.crop((int(x * L), int(y * H), int((x + w) * L), int((y + hh) * H))).save(fb)
    return Image.open(fb)


def lire(image, consigne, tentatives=TENTATIVES, max_jetons=None):
    """Renvoie ({champ: texte}, reussi). reussi = False : appel en echec, a ne pas mettre en cache."""
    service_pret()
    max_jetons = max_jetons or globals().get("MAX_JETONS", 200)
    t = io.BytesIO()
    image.convert("RGB").save(t, format="PNG")
    b64 = base64.b64encode(t.getvalue()).decode()
    for essai in range(tentatives):
        t0 = time.time()
        try:
            r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, timeout=DELAI_S, json={
                "model": MODELE, "temperature": 0, "max_tokens": max_jetons,
                "messages": [{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {"type": "text", "text": consigne}]}]})
            r.raise_for_status()
            texte = r.json()["choices"][0]["message"]["content"] or ""
            mm = re.search(r"\{.*\}", texte, re.S)
            dd = json.loads(mm.group(0)) if mm else {}
            if not mm:
                log.warning(f"reponse sans JSON ({len(texte)} car., {time.time() - t0:.1f} s)")
            return {c: str(dd.get(c, "") or "") for c in COLS}, True
        except Exception as e:
            log.warning(f"echec essai={essai + 1} {type(e).__name__}: {str(e)[:100]}")
            time.sleep(2 * (essai + 1))
    return {c: "" for c in COLS}, False


def lectures(cache, cles, fabrique, nom="lecture"):
    """Lit (ou relit depuis le cache) un ensemble de bulletins, en parallele ; renvoie cle, champ, lu.

    fabrique(cle) renvoie ({champ: texte}, reussi), ou seulement {champ: texte}.
    Sauvegarde tous les 10 bulletins et en cas d'interruption ; les echecs ne sont pas mis en
    cache (ils seront retentes a la relance) ; progression toutes les 30 s, meme sans reponse."""
    L_ = pd.read_csv(cache, dtype=str).fillna("") if cache.exists() else pd.DataFrame(columns=["cle", "champ", "lu"])
    faire = [c for c in cles if c not in set(L_.cle)]
    print(f"{nom} : {len(cles) - len(faire)} deja lus, {len(faire)} a lire ({N_PARALLELE} en parallele)", flush=True)
    log.info(f"{nom} : debut, {len(faire)} a lire")

    def une(cle):
        t0 = time.time()
        r = fabrique(cle)
        res, ok = r if isinstance(r, tuple) else (r, any(str(v).strip() for v in r.values()))
        return cle, res, ok, time.time() - t0

    rows, echecs, temps, k = [], [], [], 0
    t_debut = dernier = time.time()
    try:
        with ThreadPoolExecutor(N_PARALLELE) as ex:
            en_cours = {ex.submit(une, c) for c in faire}
            while en_cours:
                finis, en_cours = wait(en_cours, timeout=30, return_when=FIRST_COMPLETED)
                for f in finis:
                    k += 1
                    cle, res, ok, s = f.result()
                    temps.append(s)
                    if ok:
                        rows += [{"cle": cle, "champ": c, "lu": res.get(c, "")} for c in COLS]
                    else:
                        echecs.append(cle)
                if len(rows) >= 10 * len(COLS):
                    L_ = pd.concat([L_, pd.DataFrame(rows)], ignore_index=True)
                    L_.to_csv(cache, index=False)
                    rows = []
                if time.time() - dernier >= 30 or not en_cours:
                    dernier = time.time()
                    ecoule = time.time() - t_debut
                    vitesse = f"{np.mean(temps[-20:]):.1f} s/appel" if temps else "aucune reponse encore"
                    reste = f"reste ~{ecoule / k * (len(faire) - k) / 60:.0f} min" if k else ""
                    msg = f"{nom} : {k}/{len(faire)} | {ecoule / 60:.1f} min | {vitesse} | {reste} | echecs {len(echecs)}"
                    print("  " + msg, flush=True)
                    log.info(msg)
    finally:
        if rows:                                    # interruption : rien n'est perdu
            L_ = pd.concat([L_, pd.DataFrame(rows)], ignore_index=True)
            L_.to_csv(cache, index=False)
    if echecs:
        print(f"  {nom} : {len(echecs)} echecs, relancer la cellule pour les retenter")
    return L_[L_.cle.isin(set(cles))]


cache_base = RESULTATS / ("listes_insee_base.csv" if CAMPAGNE == "RG25" else f"arbitrage_{CAMPAGNE}_lectures.csv")
LECT = lectures(cache_base, list(APP.cle), lambda cle: lire(bloc(cle), PROMPT_BLOC), nom="consigne actuelle")
print(f"lectures gemma4 : {LECT.cle.nunique()} bulletins ({cache_base.name}) | journal : {JOURNAL}")

consigne actuelle : 169 deja lus, 0 a lire (4 en parallele)
lectures gemma4 : 169 bulletins (listes_insee_base.csv) | journal : /home/onyxia/work/banc-htr/resultats/journal_arbitrage_RG25.log


In [6]:
# 4. Le lecteur gemma4, et la consigne ACTUELLE (lectures du notebook 09 reutilisees)
BASE_URL = "https://llm.lab.sspcloud.fr/api"
PROMPT_BLOC = """Cette image montre un extrait de bulletin individuel du recensement francais.
Elle contient des lignes manuscrites precedees de leurs libelles imprimes :
"Nom :" et "Prenom :".

Transcris ce qui est ecrit a la main sur chacune, en MAJUSCULES, sans accent.

Regles :
- Ne corrige rien, n'ajoute rien, ne devine pas.
- Ne recopie pas les libelles imprimes, seulement l'ecriture manuscrite.
- Si une ligne n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {"NOM": "...", "PRENOM": "..."}"""
PROMPT_RELECTURE = """Cette image montre un extrait de bulletin individuel du recensement francais,
avec un nom et un prenom ecrits a la main apres les libelles imprimes "Nom :" et "Prenom :".

Une premiere lecture a donne : {lecture}
Ces valeurs ne figurent pas dans les listes de l'Insee des noms et prenoms portes en France.
Voici des valeurs existantes, proches de la premiere lecture :
{candidats}

Relis attentivement l'ecriture manuscrite.
- Si elle correspond vraiment a l'une de ces valeurs, reponds cette valeur.
- Sinon, recopie exactement ce qui est ecrit, meme si ce n'est pas dans la liste.
- Si ce n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {{"NOM": "...", "PRENOM": "..."}}"""
HEADERS = None


def service_pret():
    global HEADERS
    if HEADERS:
        return
    if "OPENAI_API_KEY" not in os.environ and shutil.which("vault"):
        r = subprocess.run(["vault", "kv", "get", "-field=OPENAI_API_KEY", "onyxia-kv/farisazibert/LLM"],
                           capture_output=True, text=True, timeout=60)
        if r.returncode == 0 and r.stdout.strip():
            os.environ["OPENAI_API_KEY"] = r.stdout.strip()
    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("Cle du service absente : la renseigner (Vault).")
    HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}


def bloc(cle):
    fb = BLOCS / f"{CAMPAGNE}_{cle}.png"
    if not fb.exists():
        chemin = APP.loc[APP.cle == cle, "chemin"].iloc[0]
        with fs.open(chemin, "rb") as src:
            page = Image.open(io.BytesIO(src.read())).convert("RGB")
        L, H = page.size
        x, y, w, hh = ZONE_BLOC
        page.crop((int(x * L), int(y * H), int((x + w) * L), int((y + hh) * H))).save(fb)
    return Image.open(fb)


def lire(image, consigne, tentatives=3):
    service_pret()
    t = io.BytesIO()
    image.convert("RGB").save(t, format="PNG")
    b64 = base64.b64encode(t.getvalue()).decode()
    for essai in range(tentatives):
        try:
            r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, timeout=180, json={
                "model": MODELE, "temperature": 0, "max_tokens": MAX_JETONS,
                "messages": [{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {"type": "text", "text": consigne}]}]})
            r.raise_for_status()
            mm = re.search(r"\{.*\}", r.json()["choices"][0]["message"]["content"] or "", re.S)
            dd = json.loads(mm.group(0)) if mm else {}
            return {c: str(dd.get(c, "") or "") for c in COLS}
        except Exception:
            time.sleep(2 * (essai + 1))
    return {c: "" for c in COLS}


def lectures(cache, cles, fabrique):
    """Lit (ou relit depuis le cache) un ensemble de bulletins ; renvoie cle, champ, lu."""
    L_ = pd.read_csv(cache, dtype=str).fillna("") if cache.exists() else pd.DataFrame(columns=["cle", "champ", "lu"])
    faire = [c for c in cles if c not in set(L_.cle)]
    rows, t0 = [], time.time()
    for k, cle in enumerate(faire):
        res = fabrique(cle)
        rows += [{"cle": cle, "champ": c, "lu": res[c]} for c in COLS]
        if (k + 1) % 25 == 0:
            print(f"  {k + 1}/{len(faire)} | {(time.time() - t0) / 60:.1f} min", flush=True)
    if rows:
        L_ = pd.concat([L_, pd.DataFrame(rows)], ignore_index=True)
        L_.to_csv(cache, index=False)
    return L_[L_.cle.isin(set(cles))]


cache_base = RESULTATS / ("listes_insee_base.csv" if CAMPAGNE == "RG25" else f"arbitrage_{CAMPAGNE}_lectures.csv")
LECT = lectures(cache_base, list(APP.cle), lambda cle: lire(bloc(cle), PROMPT_BLOC))
print(f"lectures gemma4 : {LECT.cle.nunique()} bulletins ({cache_base.name})")

lectures gemma4 : 169 bulletins (listes_insee_base.csv)


In [10]:
# 5. Les listes de l'Insee (memes regles que les notebooks 09 et 10)
LISTE = {}
for f, champ in [("noms2008nat_txt.zip", "NOM"), ("prenoms-2025-nat_csv.zip", "PRENOM")]:
    with zipfile.ZipFile(LISTES / f) as z:
        brut_l = z.read(next(n for n in z.namelist() if n.lower().endswith((".txt", ".csv"))))
    for e in ("utf-8", "latin-1"):
        try:
            texte = brut_l.decode(e)
            break
        except UnicodeDecodeError:
            continue
    sep = max(["\t", ";", ","], key=texte.split("\n", 1)[0].count)
    df = pd.read_csv(io.StringIO(texte), sep=sep, dtype=str, keep_default_na=False)
    df.columns = [x.strip().upper() for x in df.columns]
    if champ == "NOM":
        col = df.columns[0]
        n = df.drop(columns=[col]).apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
        dl = pd.DataFrame({"v": df[col].map(normaliser), "n": n})
        dl = dl[(dl.v != "") & (dl.v != "AUTRES NOMS")]
    else:
        col = next(x for x in ["PRENOM", "PREUSUEL"] if x in df.columns)
        colv = next(x for x in ["VALEUR", "NOMBRE"] if x in df.columns)
        dl = pd.DataFrame({"v": df[col].map(normaliser), "n": pd.to_numeric(df[colv], errors="coerce").fillna(0)})
        dl = dl[(dl.v != "") & ~dl.v.str.contains("PRENOMS RARES")]
    LISTE[champ] = dl.groupby("v").n.sum().to_dict()

ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ "


def dans_liste(v, c):
    if not v or v == "ILLISIBLE":
        return True
    return v in LISTE[c] or all(x in LISTE[c] for x in v.split())


def voisins1(mot):
    coupes = [(mot[:i], mot[i:]) for i in range(len(mot) + 1)]
    return set([a + b[1:] for a, b in coupes if b] + [a + b[1] + b[0] + b[2:] for a, b in coupes if len(b) > 1]
               + [a + x + b[1:] for a, b in coupes if b for x in ALPHABET] + [a + x + b for a, b in coupes for x in ALPHABET])


def candidats(v, c, ecart_max=1):
    v1 = voisins1(v)
    t = {x for x in v1 if x in LISTE[c]}
    if ecart_max >= 2 and len(t) < 5:
        for x in list(v1):
            t |= {w for w in voisins1(x) if w in LISTE[c]}
    t.discard(v)
    return sorted(t, key=lambda x: -LISTE[c][x])


def corriger(v, c, regle):
    if dans_liste(v, c) or len(v) < LONGUEUR_MIN_CORRECTION:
        return v
    k = candidats(v, c)
    if regle == "unique":
        return k[0] if len(k) == 1 else v
    return k[0] if k else v

In [9]:
# 5. Les listes de l'Insee (memes regles que les notebooks 09 et 10)
URL_NOMS = "https://www.insee.fr/fr/statistiques/fichier/3536630/noms2008nat_txt.zip"
URL_PRENOMS = "https://www.insee.fr/fr/statistiques/fichier/8595130/prenoms-2025-nat_csv.zip"
import zipfile

import requests

LISTES = Path.home() / "work" / "banc-htr" / "listes_insee"
LISTES.mkdir(parents=True, exist_ok=True)
COPIE_BUCKET = "projet-mesure-qualite-rp/references/listes_insee"
FICHIERS_LISTES = {"noms2008nat_txt.zip": globals().get("URL_NOMS"),
                   "prenoms-2025-nat_csv.zip": globals().get("URL_PRENOMS")}

# --- Recuperer les fichiers : disque local, sinon copie du bucket, sinon site de l'Insee ---
for f, url in FICHIERS_LISTES.items():
    cible = LISTES / f
    if cible.exists():
        continue
    if fs.exists(f"{COPIE_BUCKET}/{f}"):
        fs.get(f"{COPIE_BUCKET}/{f}", str(cible))
        print(f"{f} : recupere depuis le bucket")
        continue
    if not url:
        raise RuntimeError(f"{f} absent et adresse inconnue : definir URL_NOMS / URL_PRENOMS "
                           f"(memes valeurs que dans le notebook 09)")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    cible.write_bytes(r.content)
    fs.put(str(cible), f"{COPIE_BUCKET}/{f}")         # copie durable pour les prochains services
    print(f"{f} : telecharge ({len(r.content) / 1e6:.1f} Mo) et copie dans le bucket")

# --- Lecture (inchangee) ---
LISTE = {}
for f, champ in [("noms2008nat_txt.zip", "NOM"), ("prenoms-2025-nat_csv.zip", "PRENOM")]:
    with zipfile.ZipFile(LISTES / f) as z:
        brut_l = z.read(next(n for n in z.namelist() if n.lower().endswith((".txt", ".csv"))))
    for e in ("utf-8", "latin-1"):
        try:
            texte = brut_l.decode(e)
            break
        except UnicodeDecodeError:
            continue
    sep = max(["\t", ";", ","], key=texte.split("\n", 1)[0].count)
    df = pd.read_csv(io.StringIO(texte), sep=sep, dtype=str, keep_default_na=False)
    df.columns = [x.strip().upper() for x in df.columns]
    if champ == "NOM":
        col = df.columns[0]
        n = df.drop(columns=[col]).apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
        dl = pd.DataFrame({"v": df[col].map(normaliser), "n": n})
        dl = dl[(dl.v != "") & (dl.v != "AUTRES NOMS")]
    else:
        col = next(x for x in ["PRENOM", "PREUSUEL"] if x in df.columns)
        colv = next(x for x in ["VALEUR", "NOMBRE"] if x in df.columns)
        dl = pd.DataFrame({"v": df[col].map(normaliser), "n": pd.to_numeric(df[colv], errors="coerce").fillna(0)})
        dl = dl[(dl.v != "") & ~dl.v.str.contains("PRENOMS RARES")]
    LISTE[champ] = dl.groupby("v").n.sum().to_dict()
print(f"listes Insee : {len(LISTE['NOM'])} noms, {len(LISTE['PRENOM'])} prenoms")

ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ "


def dans_liste(v, c):
    if not v or v == "ILLISIBLE":
        return True
    return v in LISTE[c] or all(x in LISTE[c] for x in v.split())


def voisins1(mot):
    coupes = [(mot[:i], mot[i:]) for i in range(len(mot) + 1)]
    return set([a + b[1:] for a, b in coupes if b] + [a + b[1] + b[0] + b[2:] for a, b in coupes if len(b) > 1]
               + [a + x + b[1:] for a, b in coupes if b for x in ALPHABET] + [a + x + b for a, b in coupes for x in ALPHABET])


def candidats(v, c, ecart_max=1):
    v1 = voisins1(v)
    t = {x for x in v1 if x in LISTE[c]}
    if ecart_max >= 2 and len(t) < 5:
        for x in list(v1):
            t |= {w for w in voisins1(x) if w in LISTE[c]}
    t.discard(v)
    return sorted(t, key=lambda x: -LISTE[c][x])


def corriger(v, c, regle):
    if dans_liste(v, c) or len(v) < LONGUEUR_MIN_CORRECTION:
        return v
    k = candidats(v, c)
    if regle == "unique":
        return k[0] if len(k) == 1 else v
    return k[0] if k else v

noms2008nat_txt.zip : telecharge (3.0 Mo) et copie dans le bucket
prenoms-2025-nat_csv.zip : telecharge (4.3 Mo) et copie dans le bucket
listes Insee : 218114 noms, 43449 prenoms


In [11]:
# 6. Les nouvelles consignes, et l'image « bloc + zoom »
PROMPT_EPELER = """Cette image montre un extrait de bulletin individuel du recensement francais,
avec un nom et un prenom ecrits a la main apres les libelles imprimes "Nom :" et "Prenom :".

Transcris l'ecriture manuscrite LETTRE PAR LETTRE, en MAJUSCULES, sans accent.

Attention particuliere :
- Lettres doubles (SS, LL, RR, NN, TT, MM, FF, PP) : compte chaque lettre, n'en saute aucune.
- Lettres cursives qui se ressemblent : m/n, n/u, r/n, l/e, S/G, I/L, O/D. Regarde-les une par une.
- Si une ligne est vide, reponds VIDE. Si elle est ecrite mais illisible, reponds ILLISIBLE.
- Ne recopie pas les libelles imprimes ni l'exemple imprime.
- Ne corrige rien, ne remplace pas par un nom plus courant.

Avant de repondre, epelle chaque mot en separant les lettres par des tirets.

Reponds uniquement en JSON :
{"NOM_EPELE": "C-O-L-L-E-T", "NOM": "COLLET", "PRENOM_EPELE": "...", "PRENOM": "..."}"""

PROMPT_ZOOM = PROMPT_EPELER.replace(
    "Cette image montre un extrait de bulletin individuel du recensement francais,",
    "Cette image contient, en haut, un extrait de bulletin individuel du recensement francais, et en dessous\n"
    "un ZOOM de la ligne du nom puis un ZOOM de la ligne du prenom (memes ecritures, agrandies),")
PROMPT_ZOOM = PROMPT_ZOOM.replace("Transcris l'ecriture manuscrite LETTRE PAR LETTRE",
                                  "Utilise le haut pour savoir ou est le nom et le prenom, et les zooms pour lire chaque lettre.\n"
                                  "Transcris l'ecriture manuscrite LETTRE PAR LETTRE")

from PIL import ImageDraw as _Draw
DOSSIER_ZOOM = Path("/tmp/arbitrage/zoom")
DOSSIER_ZOOM.mkdir(parents=True, exist_ok=True)


def bloc_zoom(cle, facteur=2):
    """Une seule image : le bloc en haut, puis le zoom x2 de la ligne Nom et de la ligne Prenom."""
    f = DOSSIER_ZOOM / f"{CAMPAGNE}_{cle}.png"
    if f.exists():
        return Image.open(f)
    chemin = APP.loc[APP.cle == cle, "chemin"].iloc[0]
    with fs.open(chemin, "rb") as src_:
        page = Image.open(io.BytesIO(src_.read())).convert("RGB")
    L, H = page.size
    morceaux = [bloc(cle).convert("RGB")]
    for c in ["NOM", "PRENOM"]:
        x, y, w, h = ZONES_LIGNES[c]
        lig = page.crop((int((x - 0.01) * L), int((y - 0.004) * H), int((x + w + 0.01) * L), int((y + h + 0.004) * H)))
        lig = lig.resize((lig.width * facteur, lig.height * facteur), Image.LANCZOS)
        etiquette = Image.new("RGB", (lig.width, 34), "white")
        _Draw.Draw(etiquette).text((6, 8), f"ZOOM {c}", fill=(200, 0, 0))
        morceaux += [etiquette, lig]
    largeur = max(m.width for m in morceaux)
    planche = Image.new("RGB", (largeur, sum(m.height for m in morceaux) + 10 * len(morceaux)), "white")
    yy = 0
    for m in morceaux:
        planche.paste(m, (0, yy))
        yy += m.height + 10
    planche.save(f)
    return planche


def normaliser_reponse(v):
    v = normaliser(v)
    return "" if v == "VIDE" else v


cache_v1 = RESULTATS / f"consigne_{CAMPAGNE}_V1_epeler{SUFFIXE}.csv"
cache_v2 = RESULTATS / f"consigne_{CAMPAGNE}_V2_zoom{SUFFIXE}.csv"
print("V1 : lecture avec la consigne « epeler »...")
LECT_V1 = lectures(cache_v1, list(APP.cle), lambda cle: lire(bloc(cle), PROMPT_EPELER))
print("V2 : lecture « bloc + zoom »...")
LECT_V2 = lectures(cache_v2, list(APP.cle), lambda cle: lire(bloc_zoom(cle), PROMPT_ZOOM))
print(f"V1 : {LECT_V1.cle.nunique()} bulletins | V2 : {LECT_V2.cle.nunique()} bulletins")

V1 : lecture avec la consigne « epeler »...
  25/169 | 1.4 min
  50/169 | 2.8 min
  75/169 | 4.2 min
  100/169 | 5.6 min
  125/169 | 7.1 min
  150/169 | 8.5 min
V2 : lecture « bloc + zoom »...
  25/169 | 1.6 min
  50/169 | 3.1 min
  75/169 | 4.7 min
  100/169 | 6.4 min
  125/169 | 8.0 min
  150/169 | 9.5 min
V1 : 169 bulletins | V2 : 169 bulletins


In [12]:
# 7. La table ligne par ligne : verite arbitree, et les lectures de chaque variante
divx = {(r.cle, r.champ): r for r in DIV.itertuples()}
fiq_val = {(r.cle, c): normaliser(getattr(r, c)) for r in APP.itertuples() for c in COLS}
T = LECT[["cle", "champ"]].copy()
T["PMQ"] = [fiq_val.get((k, c), "") for k, c in zip(T.cle, T.champ)]
T = T[T.PMQ != ""].reset_index(drop=True)
verite, statut = [], []
for k, c, p in zip(T.cle, T.champ, T.PMQ):
    d = divx.get((k, c))
    if d is None:
        verite.append(p); statut.append("A")
    elif not d.arbitree:
        verite.append(None); statut.append("C")
    else:
        verite.append(normaliser(d.VALEUR_ATTENDUE)); statut.append("B")
T["verite_arbitree"], T["groupe"] = verite, statut
for nom, L_ in [("V0_actuelle", LECT), ("V1_epeler", LECT_V1), ("V2_bloc_zoom", LECT_V2)]:
    lu = {(r.cle, r.champ): normaliser_reponse(r.lu) for r in L_.itertuples()}
    T[nom] = [lu.get((k, c), "") for k, c in zip(T.cle, T.champ)]


def variante_doublee(mot):
    """Les mots obtenus en doublant une lettre, ou en retirant une lettre doublee."""
    res = set()
    for i in range(len(mot)):
        if mot[i].isalpha():
            res.add(mot[:i + 1] + mot[i] + mot[i + 1:])
        if i + 1 < len(mot) and mot[i] == mot[i + 1]:
            res.add(mot[:i] + mot[i + 1:])
    res.discard(mot)
    return res


def suspect_double(v, c):
    """Alerte : une variante a lettre doublee (ou dedoublee) existe et est bien plus portee."""
    if not v or v == "ILLISIBLE":
        return False
    for mot in v.split():
        n = LISTE[c].get(mot, 0)
        if any(LISTE[c].get(w, 0) >= max(1, n) * RATIO_DOUBLE for w in variante_doublee(mot)):
            return True
    return False


# Meilleure lecture entre V1 et V2 (jugee plus loin) : V3 = alerte sur cette lecture
def resume_simple(col, ref):
    ok = [(v == t) for v, t in zip(T[col], T[ref]) if not manquant(t)]
    return sum(ok) / max(1, len(ok))


MEILLEURE = max(["V1_epeler", "V2_bloc_zoom"], key=lambda c: resume_simple(c, "verite_arbitree"))
T["absent_liste"] = [not dans_liste(v, c) for v, c in zip(T[MEILLEURE], T.champ)]
T["suspect_double"] = [suspect_double(v, c) for v, c in zip(T[MEILLEURE], T.champ)]
T["V3_alerte_doubles"] = T[MEILLEURE]
print(f"{len(T)} lignes | meilleure lecture pour l'alerte V3 : {MEILLEURE} | "
      f"absentes des listes : {int(T.absent_liste.sum())} | suspectes lettre double : {int(T.suspect_double.sum())}")

335 lignes | meilleure lecture pour l'alerte V3 : V1_epeler | absentes des listes : 47 | suspectes lettre double : 13


In [13]:
# 8. RESULTATS : chaque variante, contre la verite arbitree et contre le Fiqual
def double_seul(lu, vrai):
    """L'erreur n'est qu'une lettre doublee en trop ou en moins."""
    return bool(lu) and lu != vrai and (lu in variante_doublee(vrai))


VARIANTES = {"V0 actuelle": ("V0_actuelle", None), "V1 epeler": ("V1_epeler", None),
             "V2 bloc + zoom": ("V2_bloc_zoom", None),
             "V0 + signal (notebook 10)": ("V0_actuelle", "v0_absent"),
             f"V3 = {MEILLEURE} + alerte (absents ou lettre double)": ("V3_alerte_doubles", "alerte")}
T["v0_absent"] = [not dans_liste(v, c) for v, c in zip(T.V0_actuelle, T.champ)]
T["alerte"] = T.absent_liste | T.suspect_double
lignes = []
for verite_nom, ref in [("arbitree", "verite_arbitree"), ("Fiqual", "PMQ")]:
    sel = T[T[ref].map(lambda v: not manquant(v))]
    base_ok = sel.V0_actuelle == sel[ref]
    pmq_ok = sel.PMQ == sel[ref]
    for nom, (col, sig) in VARIANTES.items():
        juste = sel[col] == sel[ref]
        avoue = sel[col].isin(["", "ILLISIBLE"])
        humain = avoue | (sel[sig].astype(bool) if sig else False)
        silencieux = ~juste & ~humain
        k, n = int(juste.sum()), len(sel)
        a, b, p = mcnemar(juste, base_ok)
        a2, b2, p2 = mcnemar(juste, pmq_ok)
        lignes.append({"verite": verite_nom, "variante": nom, "N": n, "justes_%": round(k / n * 100, 1), "IC95": ic(k, n),
                       "silencieuses_%": round(silencieux.mean() * 100, 1), "a_un_humain_%": round(humain.mean() * 100, 1),
                       "erreurs_lettre_double": int(sum(double_seul(l, v) for l, v in zip(sel[col], sel[ref]))),
                       "vs V0 (gagnees/perdues)": f"{a} / {b}", "McNemar p vs V0": round(p, 4),
                       "vs PMQ (seul juste / PMQ seul juste)": f"{a2} / {b2}"})
RESULTAT = pd.DataFrame(lignes)
with pd.option_context("display.max_columns", 20, "display.width", 250):
    display(RESULTAT[RESULTAT.verite == "arbitree"])
    print("\nMeme tableau contre le Fiqual brut :")
    display(RESULTAT[RESULTAT.verite == "Fiqual"])
print("\nA lire : une variante est meilleure si ses SILENCIEUSES baissent sans trop augmenter la part envoyee a un humain,"
      "\net si « vs V0 » gagne nettement plus de lignes qu'il n'en perd (McNemar p < 0,05 : ecart reel).")

,verite,variante,N,justes_%,IC95,silencieuses_%,a_un_humain_%,erreurs_lettre_double,vs V0 (gagnees/perdues),McNemar p vs V0,vs PMQ (seul juste / PMQ seul juste)
0,arbitree,V0 actuelle,318,88.7,[84.7 ; 91.7],11.3,0.0,4,0 / 0,1.000,3 / 34
1,arbitree,V1 epeler,318,89.0,[85.1 ; 92.0],11.0,0.0,1,14 / 13,1.000,2 / 32
2,arbitree,V2 bloc + zoom,318,86.5,[82.3 ; 89.8],13.5,0.0,0,15 / 22,0.324,2 / 40
3,arbitree,V0 + signal (notebook 10),318,88.7,[84.7 ; 91.7],5.7,12.3,4,0 / 0,1.000,3 / 34
4,arbitree,V3 = V1_epeler + alerte (absents ou lettre dou...,318,89.0,[85.1 ; 92.0],4.7,14.2,1,14 / 13,1.000,2 / 32



Meme tableau contre le Fiqual brut :


,verite,variante,N,justes_%,IC95,silencieuses_%,a_un_humain_%,erreurs_lettre_double,vs V0 (gagnees/perdues),McNemar p vs V0,vs PMQ (seul juste / PMQ seul juste)
5,Fiqual,V0 actuelle,335,84.5,[80.2 ; 88.0],14.6,0.9,3,0 / 0,1.0000,0 / 52
6,Fiqual,V1 epeler,335,85.4,[81.2 ; 88.8],14.0,0.6,0,16 / 13,0.7111,0 / 49
7,Fiqual,V2 bloc + zoom,335,84.2,[79.9 ; 87.7],15.8,0.0,1,20 / 21,1.0000,0 / 53
8,Fiqual,V0 + signal (notebook 10),335,84.5,[80.2 ; 88.0],7.5,14.6,3,0 / 0,1.0000,0 / 52
9,Fiqual,V3 = V1_epeler + alerte (absents ou lettre dou...,335,85.4,[81.2 ; 88.8],5.7,16.4,0,16 / 13,0.7111,0 / 49



A lire : une variante est meilleure si ses SILENCIEUSES baissent sans trop augmenter la part envoyee a un humain,
et si « vs V0 » gagne nettement plus de lignes qu'il n'en perd (McNemar p < 0,05 : ecart reel).


In [14]:
# 9. Les lignes qui changent : ce que chaque variante a gagne ou perdu (a verifier sur l'image)
CHG = T[(T.V0_actuelle != T.V1_epeler) | (T.V0_actuelle != T.V2_bloc_zoom)].copy()
CHG["verdict_V0"] = ["juste" if v == t else "faux" for v, t in zip(CHG.V0_actuelle, CHG.verite_arbitree)]
CHG["verdict_V1"] = ["juste" if v == t else "faux" for v, t in zip(CHG.V1_epeler, CHG.verite_arbitree)]
CHG["verdict_V2"] = ["juste" if v == t else "faux" for v, t in zip(CHG.V2_bloc_zoom, CHG.verite_arbitree)]
CHG["verite_arbitree"] = CHG.verite_arbitree.map(lambda v: "(inconnue)" if manquant(v) else v)
COLS_CHG = ["cle", "champ", "groupe", "verite_arbitree", "V0_actuelle", "verdict_V0", "V1_epeler", "verdict_V1",
            "V2_bloc_zoom", "verdict_V2", "suspect_double"]
with pd.option_context("display.max_rows", 60, "display.width", 250):
    display(CHG[COLS_CHG].head(60))
print(f"{len(CHG)} lignes changent entre les variantes. Images « bloc + zoom » : {DOSSIER_ZOOM}")

,cle,champ,groupe,verite_arbitree,V0_actuelle,verdict_V0,V1_epeler,verdict_V1,V2_bloc_zoom,verdict_V2,suspect_double
5,6901_5300000222,NOM,B,PEREZ JEAN,PEREZ JEAN,juste,PEREZ,faux,PEREZ,faux,False
6,6901_5300000222,PRENOM,C,(inconnue),ILLISIBLE,faux,JEAN,faux,JEAN,faux,False
10,6901_5300000311,PRENOM,A,ANTOINE,ANTOME,faux,ANTONE,faux,ANTOME,faux,False
24,6901_5300000342,PRENOM,A,ERIC,ERIC,juste,ERIC,juste,ENIC,faux,False
26,6901_5300000339,PRENOM,A,CEDRIC,CELINE,faux,CELINE,faux,CEDRIC,juste,False
30,6901_5300000340,PRENOM,A,MILENA,M LENA,faux,M LENA,faux,MLENA,faux,False
31,6901_5300000344,NOM,A,MARTIN,MARTIN,juste,MARCHET,faux,MAURIN,faux,False
36,6901_5300000346,PRENOM,A,PIERRE,PIERRE,juste,PIERRE,juste,PRENE,faux,False
38,6901_5300000349,PRENOM,A,MARC,MARC,juste,MARE,faux,MARC,juste,False
44,6901_5300000354,PRENOM,A,CEDRIC,CELINE,faux,CEDRIC,juste,CEDRIC,juste,False


64 lignes changent entre les variantes. Images « bloc + zoom » : /tmp/arbitrage/zoom


In [15]:
# 10. Enregistrement : Excel, et MLflow (personnel + partage)
f_xlsx = RESULTATS / f"consigne_{CAMPAGNE}_variantes{SUFFIXE}.xlsx"
with pd.ExcelWriter(f_xlsx) as w:
    RESULTAT.to_excel(w, sheet_name="1_resultats", index=False)
    CHG[COLS_CHG].to_excel(w, sheet_name="2_lignes_qui_changent", index=False)
    T.to_excel(w, sheet_name="3_toutes_les_lignes", index=False)
print("Excel :", f_xlsx)


def enregistrer():
    with mlflow.start_run(run_name=f"consigne-variantes-{CAMPAGNE}") as run:
        mlflow.set_tags({"etude": "ameliorer-consigne", "notebook": "ameliorer_consigne", "statut": "exploratoire",
                         "campagne": CAMPAGNE, "modele": MODELE})
        mlflow.log_params({"lignes": len(T), "ratio_double": RATIO_DOUBLE, "meilleure_lecture_v3": MEILLEURE})
        for _, r in RESULTAT.iterrows():
            nom = re.sub(r"[^a-z0-9]+", "_", f"{r['variante']}_{r['verite']}".lower()).strip("_")[:80]
            mlflow.log_metrics({f"{nom}_justes_pct": float(r["justes_%"]),
                                f"{nom}_silencieuses_pct": float(r["silencieuses_%"]),
                                f"{nom}_humain_pct": float(r["a_un_humain_%"])})
        mlflow.log_text(PROMPT_EPELER, "consigne_V1_epeler.txt")
        mlflow.log_text(PROMPT_ZOOM, "consigne_V2_zoom.txt")
        mlflow.log_artifact(str(f_xlsx))
    return run.info.run_id[:8]


if TEST_RAPIDE:
    print("Mode TEST RAPIDE : rien n'est enregistre dans MLflow.")
else:
    try:
        import importlib
        sys.path.insert(0, str(RACINE / "src"))
        from mesure_qualite.evaluation import double_mlflow as dm
        dm = importlib.reload(dm)
        print(dm.partout(enregistrer))
    except ImportError:
        mlflow.set_experiment("fiqual-banc-essai")
        print("MLflow personnel seulement :", enregistrer())

Excel : /home/onyxia/work/banc-htr/resultats/consigne_RG25_variantes.xlsx
🏃 View run consigne-variantes-RG25 at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/6096199842034d829daaf8a8e26fce0d
🧪 View experiment at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/1
MLflow personnel seulement : 60961998


In [24]:
# 10. Enregistrement : Excel (local), et MLflow (personnel + partage), comme le notebook de reference
import importlib

f_xlsx = RESULTATS / f"consigne_{CAMPAGNE}_variantes{SUFFIXE}.xlsx"
with pd.ExcelWriter(f_xlsx) as w:
    RESULTAT.to_excel(w, sheet_name="1_resultats", index=False)
    CHG[COLS_CHG].to_excel(w, sheet_name="2_lignes_qui_changent", index=False)
    T.to_excel(w, sheet_name="3_toutes_les_lignes", index=False)
print("Excel (local) :", f_xlsx)

# Pour MLflow : la synthese seule (l'Excel contient les lectures, donc des noms)
f_synthese = RESULTATS / f"consigne_{CAMPAGNE}_variantes_synthese{SUFFIXE}.csv"
RESULTAT.to_csv(f_synthese, index=False)


def enregistrer():
    with mlflow.start_run(run_name=f"consigne-variantes-{CAMPAGNE}") as run:
        mlflow.set_tags({"etude": "ameliorer-consigne", "notebook": "ameliorer_consigne", "statut": "exploratoire",
                         "campagne": CAMPAGNE, "modele": MODELE})
        mlflow.log_params({"lignes": len(T), "ratio_double": RATIO_DOUBLE, "meilleure_lecture_v3": MEILLEURE})
        for _, r in RESULTAT.iterrows():
            nom = re.sub(r"[^a-z0-9]+", "_", f"{r['variante']}_{r['verite']}".lower()).strip("_")[:80]
            for col, suffixe in [("justes_%", "justes_pct"), ("silencieuses_%", "silencieuses_pct"),
                                 ("a_un_humain_%", "humain_pct")]:
                v = pd.to_numeric(r[col], errors="coerce")
                if pd.notna(v):
                    mlflow.log_metric(f"{nom}_{suffixe}", float(v))
        mlflow.log_text(PROMPT_EPELER, "consigne_V1_epeler.txt")
        mlflow.log_text(PROMPT_ZOOM, "consigne_V2_zoom.txt")
        mlflow.log_artifact(str(f_synthese))
    return run.info.run_id[:8]


if TEST_RAPIDE:
    print("Mode TEST RAPIDE : rien n'est enregistre dans MLflow.")
else:
    sys.path.insert(0, str(RACINE / "src"))       # reprendre ici les memes chemins que le notebook de reference
    try:
        from mesure_qualite.evaluation import double_mlflow as dm
        dm = importlib.reload(dm)
    except ImportError as e:
        raise ImportError(f"double_mlflow introuvable ({e}). Reprendre l'import du notebook de reference.") from e
    print(dm.partout(enregistrer))

Excel (local) : /home/onyxia/work/banc-htr/resultats/consigne_RG25_variantes.xlsx
🏃 View run consigne-variantes-RG25 at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/5cd01a45f2c5418494c899e98a859324
🧪 View experiment at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/1
MLflow perso    : enregistre (https://user-farisazibert-mlflow.user.lab.sspcloud.fr)
🏃 View run consigne-variantes-RG25 at: https://projet-mesure-qualite-rp-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/fcc49b396a52457f871b926d31ce8406
🧪 View experiment at: https://projet-mesure-qualite-rp-mlflow.user.lab.sspcloud.fr/#/experiments/1
MLflow partage  : enregistre (https://projet-mesure-qualite-rp-mlflow.user.lab.sspcloud.fr/)
{'perso': '5cd01a45', 'partage': 'fcc49b39'}
